In [8]:
import ee
import geemap.core as geemap
import os

In [9]:
# Login to GEE
ee.Authenticate()
ee.Initialize(project='ee-cbra1395') # Your project name here

In [10]:
# Define your parameters
center = ee.Geometry.Point([-118.9689,34.3189])
dataset_name = "Extracted Data"


california = ee.FeatureCollection("TIGER/2018/States").filter(ee.Filter.eq('NAME', 'California'))

if not os.path.exists(dataset_name):
    os.mkdir(dataset_name)

datasets = {
    dataset_name:ee.ImageCollection("LANDSAT/LC09/C02/T1")
      .filterDate('2024-11-06', '2024-11-22') # Adjust date range, ensure data is avaliable in this range or it cannot export. Note the 16 day revist interval of Landsat 9
      .filterBounds(california),
}


# Rectangle for grid in degrees
height = 0.1
width = 2.5 * height

lon = ee.Number(center.coordinates().get(0))
lat = ee.Number(center.coordinates().get(1))

xmin = lon.subtract(width / 2)
xmax = lon.add(width / 2)
ymin = lat.subtract(height / 2)
ymax = lat.add(height / 2)

rectangle = ee.Geometry.Rectangle([xmin, ymin, xmax, ymax])
region = rectangle

grid = ee.FeatureCollection(region.coveringGrid('EPSG:4326', scale=2500))
grid_list = grid.toList(grid.size())
grid_size = grid.size().getInfo()

for dataset_names, dataset in datasets.items():
    median_image = dataset.median()

    for i in range(grid_size):
        # Get tile info
        region_tile = ee.Feature(grid_list.get(i)).geometry()
        coords = ee.List(region_tile.bounds().coordinates().get(0))
        top_left = ee.List(coords.get(0))
        bottom_right = ee.List(coords.get(2))
        tile_name = f"TL_{top_left.get(1).getInfo():.5f}_{top_left.get(0).getInfo():.5f}__BR_{bottom_right.get(1).getInfo():.5f}_{bottom_right.get(0).getInfo():.5f}" #Tile top left and bottom right coordinates to 5 decimal places in tile name

        print(f"Exporting tile {i} - Extracted_Tile_{tile_name}")

        task = ee.batch.Export.image.toDrive(
            image=median_image.clip(region_tile),
            description=f'Extracted_Tile_{i+1}_{tile_name}',
            folder=dataset_names,
            region=region_tile,
            scale=30,
            maxPixels=1e10
        )
        task.start()

Exporting tile 0 - Extracted_Tile_TL_34.24827_-119.09415__BR_34.27073_-119.07169
Exporting tile 1 - Extracted_Tile_TL_34.24827_-119.07169__BR_34.27073_-119.04923
Exporting tile 2 - Extracted_Tile_TL_34.24827_-119.04923__BR_34.27073_-119.02678
Exporting tile 3 - Extracted_Tile_TL_34.24827_-119.02678__BR_34.27073_-119.00432
Exporting tile 4 - Extracted_Tile_TL_34.24827_-119.00432__BR_34.27073_-118.98186
Exporting tile 5 - Extracted_Tile_TL_34.24827_-118.98186__BR_34.27073_-118.95940
Exporting tile 6 - Extracted_Tile_TL_34.24827_-118.95940__BR_34.27073_-118.93694
Exporting tile 7 - Extracted_Tile_TL_34.24827_-118.93694__BR_34.27073_-118.91449
Exporting tile 8 - Extracted_Tile_TL_34.24827_-118.91449__BR_34.27073_-118.89203
Exporting tile 9 - Extracted_Tile_TL_34.24827_-118.89203__BR_34.27073_-118.86957
Exporting tile 10 - Extracted_Tile_TL_34.24827_-118.86957__BR_34.27073_-118.84711
Exporting tile 11 - Extracted_Tile_TL_34.24827_-118.84711__BR_34.27073_-118.82465
Exporting tile 12 - Extrac